# 01 · Hiperparametre ayarlama (eşit bütçe, seçim yalnız NN val)
Her model için 8 kombinasyon (`GRIDS`), tek ayarlama seed'i (`TUNE_SEED = 7`; final seed'lerinden farklı), en fazla `TUNE_EPOCHS` epoch,
erken durma `TUNE_PATIENCE`. Seçim: en düşük NN val RMSE. Test bu aşamada **hiç** kullanılmaz. Sonuç → `results\configs\selected.json`.

In [ ]:
import sys, os, json, time, warnings
import os; ROOT = os.path.abspath('..')  # repository root (run notebooks from the notebooks/ folder)
SMOKE = False                 # True: çok küçük hızlı deneme (kurulum testi)  False
sys.path.insert(0, ROOT)
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch
from data_utils import *
from models.experiment import *
from models import engine
exp = Experiment(ROOT, smoke=SMOKE)
exp.load_data()
print('Cihaz:', exp.dev, '| W =', exp.cfg['W'])

In [ ]:
rows = []
for name in GRIDS:
    combos = grid_specs(name)[:2] if exp.smoke else grid_specs(name)
    for tag, spec in combos:
        rec = exp.run(tag, spec, exp.cfg['TUNE_SEED'], 'tune', evaluate=False, tune=True)
        tr = rec['train']; s = rec['spec']
        rows.append({'model': name, 'kombinasyon': tag, 'lr': s['lr'], **{f'p.{k}': v for k, v in s['params'].items()},
                     'parametre': rec['n_params'], 'val_RMSE_%': tr['best_val'] * 100, 'en iyi epoch': tr['best_epoch'],
                     'epoch sayısı': tr['epochs_run'], 'süre_dk': tr['train_time_s'] / 60, 'spec': json.dumps(s, ensure_ascii=False)})
tune = pd.DataFrame(rows)
sel = {}
for name, g in tune.groupby('model', sort=False):
    b = g.loc[g['val_RMSE_%'].idxmin()]; sel[name] = json.loads(b.spec)
    print(f"{name:28s} seçilen: {b.kombinasyon:40s} val {b['val_RMSE_%']:.3f}%")
exp.save_selected(sel)
with pd.ExcelWriter(exp.p['excel'] / '01_ayarlama.xlsx') as xw:
    tune.drop(columns='spec').to_excel(xw, sheet_name='Tüm kombinasyonlar', index=False)
    tune.loc[tune.groupby('model', sort=False)['val_RMSE_%'].idxmin()].drop(columns='spec').to_excel(xw, sheet_name='Seçilenler', index=False)
    pd.DataFrame([{'toplam ayarlama süresi (saat)': tune['süre_dk'].sum() / 60, 'koşu': len(tune)}]).to_excel(xw, sheet_name='Bütçe', index=False)
print(f"Toplam ayarlama: {tune['süre_dk'].sum() / 60:.1f} saat")